# Figure S1 Unified Data Table

This notebook builds one final table by combining:
- `batch_summary.csv` (primary source)
- paired-trial metadata used in Figure 2 (`trial4/4.5`, `trial5/5.5`, `trial6/6.5`, `trial7/7.5`)
- `signal_rate_summary.csv` values merged onto matching trial/brick/channel rows

Output: `data/processed/FigureS1_unified_data_table.csv`.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

def first_existing(paths):
    return next((p.resolve() for p in paths if p.exists()), None)

batch_candidates = [
    Path.cwd() / '../data/processed/batch_summary.csv',
    Path.cwd() / 'data/processed/batch_summary.csv',
    Path('/Users/oakley/Documents/GitHub/signal_respirometry/data/processed/batch_summary.csv'),
]
signal_candidates = [
    Path.cwd() / '../data/processed/signal_rate_summary.csv',
    Path.cwd() / 'data/processed/signal_rate_summary.csv',
    Path('/Users/oakley/Documents/GitHub/signal_respirometry/data/processed/signal_rate_summary.csv'),
]

batch_path = first_existing(batch_candidates)
signal_path = first_existing(signal_candidates)

if batch_path is None:
    raise FileNotFoundError('Could not find data/processed/batch_summary.csv')
if signal_path is None:
    raise FileNotFoundError('Could not find data/processed/signal_rate_summary.csv')

batch_df = pd.read_csv(batch_path)
signal_df = pd.read_csv(signal_path)

print(f'Loaded batch summary: {batch_path} ({len(batch_df)} rows)')
print(f'Loaded signal summary: {signal_path} ({len(signal_df)} rows)')
display(batch_df.head())
display(signal_df.head())

Loaded batch summary: /Users/oakley/Documents/GitHub/signal_respirometry/data/processed/batch_summary.csv (71 rows)
Loaded signal summary: /Users/oakley/Documents/GitHub/signal_respirometry/data/processed/signal_rate_summary.csv (11 rows)


,trial,brick,channel,n,total_mass_mg,corrected_mass_mg,RMR,A,M,temp_C,vessel,date,environment,notes,filtered
0,trial1,box2,Ch2,1.0,0.2263,NaN,2.708969,1.868423,2.644998,27.948,small,10Nov2025,night,NaN,True
1,trial1,box2,Ch3,1.0,0.2099,NaN,3.669335,2.483650,3.515933,27.948,small,10Nov2025,night,NaN,True
2,trial1,box2,Ch4,1.0,0.2703,NaN,2.927729,2.111020,2.988427,27.948,small,10Nov2025,night,NaN,True
3,trial1,box3,Ch2,20.0,7.6000,7.238095,8.546660,6.628954,9.384154,28.233,cylinder,10Nov2025,night,NaN,True
4,trial1,box3,Ch3,20.0,6.5000,7.222222,8.352764,6.475010,9.166227,28.233,cylinder,10Nov2025,night,NaN,True


,night,position,signal_rate,trial,brick,channel
0,2025Nov10,left,4.161491,trial1,box3,Ch4
1,2025Nov10,middle,0.981366,trial1,box3,Ch3
2,2025Nov10,right,9.745342,trial1,box3,Ch2
3,2025Nov12,left,0.233474,trial3,box2,Ch4
4,2025Nov12,middle,0.485232,trial3,box2,Ch3


In [7]:
# Paired-trial rules from Figure2SlopeGraph.ipynb
pair_trials = [
    ('trial4', 'trial4.5'),
    ('trial5', 'trial5.5'),
    ('trial6', 'trial6.5'),
    ('trial7', 'trial7.5'),
]
pair_lookup = {}
for a, b in pair_trials:
    pair_name = f'{a}_vs_{b}'
    pair_lookup[a] = {'pair_name': pair_name, 'pair_partner_trial': b}
    pair_lookup[b] = {'pair_name': pair_name, 'pair_partner_trial': a}

unified = batch_df.copy()

# Add paired-trial metadata columns.
unified['is_paired_trial'] = unified['trial'].isin(pair_lookup)
unified['pair_name'] = unified['trial'].map(lambda t: pair_lookup.get(t, {}).get('pair_name'))
unified['pair_partner_trial'] = unified['trial'].map(lambda t: pair_lookup.get(t, {}).get('pair_partner_trial'))

# For paired rows, test whether this exact brick+channel has both day and night entries
# across the two linked trials (same logic shape as Figure 2 pairing).
pair_env_counts = (
    unified[unified['is_paired_trial']]
    .groupby(['pair_name', 'brick', 'channel'])['environment']
    .agg(lambda s: len(set(s.dropna())))
    .rename('pair_environment_count')
    .reset_index()
)
pair_env_counts['pair_has_day_night_for_brick_channel'] = pair_env_counts['pair_environment_count'] >= 2

unified = unified.merge(
    pair_env_counts[['pair_name', 'brick', 'channel', 'pair_has_day_night_for_brick_channel']],
    on=['pair_name', 'brick', 'channel'],
    how='left',
)
pair_flag = unified['pair_has_day_night_for_brick_channel']
unified['pair_has_day_night_for_brick_channel'] = np.where(pair_flag.isna(), False, pair_flag).astype(bool)

# Robust key normalization for signal-rate joining.
def norm_key(series):
    return series.astype(str).str.strip().str.lower()

unified['_trial_k'] = norm_key(unified['trial'])
unified['_brick_k'] = norm_key(unified['brick'])
unified['_channel_k'] = norm_key(unified['channel'])

signal_work = signal_df.copy()
signal_work['_trial_k'] = norm_key(signal_work['trial'])
signal_work['_brick_k'] = norm_key(signal_work['brick'])
signal_work['_channel_k'] = norm_key(signal_work['channel'])

signal_keyed = signal_work.groupby(['_trial_k', '_brick_k', '_channel_k'], as_index=False)['signal_rate'].mean()

unified = unified.merge(
    signal_keyed,
    on=['_trial_k', '_brick_k', '_channel_k'],
    how='left',
    validate='many_to_one',
)

# Coverage diagnostics focused on filtered cylinder rows (Figure 3-relevant rows).
filtered_cyl = unified[(unified['filtered'] == True) & (unified['vessel'] == 'cylinder')]
n_filtered_cyl = len(filtered_cyl)
n_filtered_cyl_with_signal = filtered_cyl['signal_rate'].notna().sum()

print('Paired-trial rows:', int(unified['is_paired_trial'].sum()))
print('Unique trial-pair blocks represented:', unified['pair_name'].dropna().nunique())
print(f'Filtered cylinder rows with signal_rate: {n_filtered_cyl_with_signal}/{n_filtered_cyl}')

missing_filtered_signal = filtered_cyl[filtered_cyl['signal_rate'].isna()]
if missing_filtered_signal.empty:
    print('No filtered cylinder rows are missing signal_rate.')
else:
    print('Filtered cylinder rows still missing signal_rate (inspect):')
    display(missing_filtered_signal[['trial', 'brick', 'channel', 'environment', 'filtered']])

# Keep a clean column order with new columns near the end.
base_cols = [c for c in batch_df.columns if c in unified.columns]
new_cols = [
    'signal_rate',
    'is_paired_trial',
    'pair_name',
    'pair_partner_trial',
    'pair_has_day_night_for_brick_channel',
]
final_cols = base_cols + [c for c in new_cols if c not in base_cols]
unified_final = unified[final_cols].copy()

# Display full table in notebook output.
display(unified_final)

Paired-trial rows: 50
Unique trial-pair blocks represented: 4
Filtered cylinder rows with signal_rate: 8/8
No filtered cylinder rows are missing signal_rate.


,trial,brick,channel,n,total_mass_mg,corrected_mass_mg,RMR,A,M,temp_C,vessel,date,environment,notes,filtered,signal_rate,is_paired_trial,pair_name,pair_partner_trial,pair_has_day_night_for_brick_channel
0,trial1,box2,Ch2,1.0,0.2263,NaN,2.708969,1.868423,2.644998,27.948,small,10Nov2025,night,NaN,True,NaN,False,None,None,False
1,trial1,box2,Ch3,1.0,0.2099,NaN,3.669335,2.483650,3.515933,27.948,small,10Nov2025,night,NaN,True,NaN,False,None,None,False
2,trial1,box2,Ch4,1.0,0.2703,NaN,2.927729,2.111020,2.988427,27.948,small,10Nov2025,night,NaN,True,NaN,False,None,None,False
3,trial1,box3,Ch2,20.0,7.6000,7.238095,8.546660,6.628954,9.384154,28.233,cylinder,10Nov2025,night,NaN,True,9.745342,False,None,None,False
4,trial1,box3,Ch3,20.0,6.5000,7.222222,8.352764,6.475010,9.166227,28.233,cylinder,10Nov2025,night,NaN,True,0.981366,False,None,None,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,trial7.5,newbox,Ch3,NaN,0.2633,NaN,2.798130,2.004383,2.837467,26.549,small,17Nov2025,night,Same animals as trial7; Ch1 control in same CSV,True,NaN,True,trial7_vs_trial7.5,trial7,True
67,trial7.5,newbox,Ch4,NaN,0.2362,NaN,2.626962,1.831360,2.592531,26.549,small,17Nov2025,night,Same animals as trial7; Ch1 control in same CSV,True,NaN,True,trial7_vs_trial7.5,trial7,True
68,trial7.5,box3,Ch2,NaN,0.2644,NaN,1.358387,0.974067,1.378920,26.417,small,17Nov2025,night,Same animals as trial7; Ch1 control from newpy...,True,NaN,True,trial7_vs_trial7.5,trial7,True
69,trial7.5,box3,Ch3,NaN,0.2535,NaN,1.360360,0.965269,1.366465,26.417,small,17Nov2025,night,Same animals as trial7; Ch1 control from newpy...,True,NaN,True,trial7_vs_trial7.5,trial7,True


In [6]:
out_path = (Path.cwd() / '../data/processed/FigureS1_unified_data_table.csv').resolve()
out_path.parent.mkdir(parents=True, exist_ok=True)
unified_final.to_csv(out_path, index=False)

print(f'Saved unified table: {out_path}')
print(f'Rows: {len(unified_final)} | Columns: {len(unified_final.columns)}')
print('New columns added: signal_rate, is_paired_trial, pair_name, pair_partner_trial, pair_has_day_night_for_brick_channel')

Saved unified table: /Users/oakley/Documents/GitHub/signal_respirometry/data/processed/FigureS1_unified_data_table.csv
Rows: 71 | Columns: 20
New columns added: signal_rate, is_paired_trial, pair_name, pair_partner_trial, pair_has_day_night_for_brick_channel
